# Практическая работа №1 — вариант 1
## Табличные агенты для навигации дрона

**Источник:** `1_Tabular.pdf`, разделы 2–5 и 7. **Сравнение:** рациональный Q-learning, Prospect Theory Q-learning и энтропийный risk-sensitive Q-learning. Запускайте ячейки последовательно в Google Colab. Исходный пример `Lab_1_AI_agent.ipynb` сохранён без изменений.

**Оговорки к методичке.** Указаны 16 действий, но явно определены только 4 направления × 3 скорости = 12. Здесь добавлены 4 служебных действия «удержание позиции» с целевыми скоростями 0, 0.10, 0.25, 0.40; вектор направления для них не используется. Это явное проектное решение, а не спецификация из PDF. Порог успеха — заряд >0.90 до шага 140, поэтому для зарядки нужно задержаться на станции. Для теста при ε=0.005 используем *реальный* ε-greedy, а не жадный режим. В PDF не указан штраф за столкновение: выбран запрет входа, без отдельного штрафа, только базовая пошаговая награда. Отчётные значения вычисляются при выполнении, а не выдумываются.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from dataclasses import dataclass
from collections import defaultdict
from itertools import combinations

SEED = 2026
TRAIN_EPISODES = 15_000
TEST_EPISODES = 300
GAMMA = 0.99
ALPHA = 0.1
EPS_DECAY = 0.997
ETA = 0.5  # разрешённое значение; более мягкая риск-аверсия и устойчивее экспонента
np.set_printoptions(precision=4, suppress=True)
print("NumPy:", np.__version__, "| seed:", SEED)

## 1. MDP и реализация среды

Состояние: (Δx, Δy, b, v), где Δx и Δy — подписанные смещения от дрона до станции. Для координатных бинов используем диапазоны [-24,24] и [-12,12]; батарея — 12 бинов, скорость — 4. Итого 14×14×12×4 = 9408 дискретных состояний. Ветер фиксирован в пределах 15 шагов. При пересечении отрезком движения препятствия позиция не изменяется, но расход энергии и время учитываются. Ветер на шаге 0 генерируется при reset. Эпизод завершён по успеху, разряду или 200 шагам; достижение станции после шага 140 не засчитывается успехом.

In [ ]:
@dataclass(frozen=True)
class Config:
    width: float = 24.
    height: float = 12.
    station: tuple = (21.5, 8.5)
    radius: float = 0.9
    charge: float = 0.025
    wind_period: int = 15
    max_steps: int = 200
    success_deadline: int = 140
    obstacles: tuple = ((10.,11.5,2.,5.5),(13.,14.5,7.,10.),
                        (16.5,18.,3.5,6.5),(19.,20.5,9.,11.5))
    speeds: tuple = (0.10,0.25,0.40,0.60)

CFG = Config()

def segment_hits_rect(p, q, rect):
    # Liang–Barsky: пересечение закрытого прямоугольника отрезком
    xmin,xmax,ymin,ymax = rect
    dx,dy = q[0]-p[0],q[1]-p[1]
    t0,t1 = 0.,1.
    for pp,qq in ((-dx,p[0]-xmin),(dx,xmax-p[0]),(-dy,p[1]-ymin),(dy,ymax-p[1])):
        if abs(pp) < 1e-12:
            if qq < 0: return False
        else:
            r = qq/pp
            if pp < 0: t0 = max(t0,r)
            else: t1 = min(t1,r)
            if t0 > t1: return False
    return True

class DroneEnv:
    def __init__(self, seed=SEED):
        self.cfg = CFG
        self.rng = np.random.default_rng(seed)
        self.actions = [(angle,speed,False) for angle in (0,np.pi/2,np.pi,3*np.pi/2)
                        for speed in CFG.speeds[:3]]
        self.actions += [(0.,speed,True) for speed in (0.,*CFG.speeds[:3])]
        self.n_actions = len(self.actions)
        self.reset()

    def observe(self):
        dx,dy = np.array(self.cfg.station) - self.pos
        coords = [np.clip(dx,-24,24),np.clip(dy,-12,12)]
        bx = int(np.clip(np.floor((coords[0]+24)/48*14),0,13))
        by = int(np.clip(np.floor((coords[1]+12)/24*14),0,13))
        bb = int(np.clip(np.floor(self.battery*12),0,11))
        bv = int(np.argmin(abs(np.array(self.cfg.speeds)-self.speed)))
        return (bx,by,bb,bv)

    def reset(self):
        xy = self.rng.normal((3.5,6.0),1.8,2)
        self.pos = np.clip(xy,[1.,2.],[7.,10.]).astype(float)
        self.battery = 1.
        self.speed = 0.10
        self.t = 0
        self.wind = self.rng.uniform(-0.11,0.11,2)
        return self.observe()

    def step(self, action):
        angle,target,hold = self.actions[int(action)]
        # Скорость 0.60 доступна в состоянии, но не как команда: инерционная модель
        # с 25% вероятностью краткого разгона на быстром манёвре.
        if hold:
            v = target
            delta = np.zeros(2)
        else:
            v = 0.60 if target == 0.40 and self.rng.random() < 0.25 else target
            delta = v*np.array([np.cos(angle),np.sin(angle)])
        if self.t and self.t % self.cfg.wind_period == 0:
            self.wind = self.rng.uniform(-0.11,0.11,2)
        old = self.pos.copy()
        new = np.clip(old + delta + self.wind,[0.,0.],[24.,12.])
        collision = any(segment_hits_rect(old,new,r) for r in self.cfg.obstacles)
        if not collision: self.pos = new
        self.speed = v if v > 0 else 0.10  # 0 отсутствует в допустимом множестве v
        inside = np.linalg.norm(self.pos-np.array(self.cfg.station)) < self.cfg.radius
        self.battery = float(np.clip(self.battery - 0.018*v*v
                          - 0.003*np.dot(self.wind,self.wind) - 0.002*abs(v)
                          + (self.cfg.charge if inside else 0.),0.,1.))
        self.t += 1
        success = bool(inside and self.battery > 0.90 and self.t <= self.cfg.success_deadline)
        depleted = bool(self.battery < 0.04)
        timeout = self.t >= self.cfg.max_steps
        done = success or depleted or timeout
        reward = (90. if success else -170. if depleted else
                  -1.0-0.030*v*v+0.12*self.cfg.charge*int(inside))
        return self.observe(),reward,done,dict(success=success,depleted=depleted,
            timeout=timeout,collision=bool(collision),inside=bool(inside),
            battery=self.battery,position=self.pos.copy(),wind=self.wind.copy())

**Динамическая скорость и служебные действия** не детализированы в PDF. Здесь явно выбрана простая стохастическая модель ускорения; для всех агентов она одинакова. Служебное «удержание» подвержено ветровому сносу. Для полной воспроизводимости у каждой среды свой генератор случайных чисел.

In [ ]:
class TabularAgent:
    def __init__(self, n_actions=16, seed=SEED):
        self.q = defaultdict(lambda: np.zeros(n_actions,dtype=float))
        self.n_actions = n_actions
        self.rng = np.random.default_rng(seed)
    def values(self,s): return self.q[s]
    def act(self,s,epsilon):
        if self.rng.random() < epsilon: return int(self.rng.integers(self.n_actions))
        vals = self.values(s)
        best = np.flatnonzero(np.isclose(vals,vals.max()))
        return int(self.rng.choice(best))
    def transform(self,r): return r
    def update(self,s,a,r,ns,done):
        target = self.transform(r)+(0. if done else GAMMA*np.max(self.values(ns)))
        self.q[s][a] += ALPHA*(target-self.q[s][a])

class ProspectAgent(TabularAgent):
    def transform(self,r):
        return r**0.88 if r >= 0 else -2.35*(-r)**0.88

class RiskAgent(TabularAgent):
    def __init__(self,n_actions=16,seed=SEED,eta=ETA):
        super().__init__(n_actions,seed)
        self.eta = eta
        self.u = defaultdict(lambda: np.ones(n_actions,dtype=float))
    def values(self,s):
        return -np.log(np.maximum(self.u[s],1e-300))/self.eta
    def update(self,s,a,r,ns,done):
        # Уравнение (19) и обновление (20); терминальный множитель равен 1
        continuation = 1. if done else np.min(self.u[ns])**GAMMA
        target = np.exp(np.clip(-self.eta*r,-700,700))*continuation
        self.u[s][a] = max(1e-300,(1-ALPHA)*self.u[s][a]+ALPHA*target)

## 2. Обучение и проверка

15 000 эпизодов на каждого агента, ε₀=1, εmin=0.01, εdecay=0.997, γ=0.99, α=0.1. Сохраняются награда, длина, коллизии и финальный заряд. Тестирование: 300 эпизодов, ε=0.005. Одинаковые seed среды для разных агентов задают сопоставимые потоки случайных чисел, но из-за различающихся действий траектории RNG могут расходиться; это не строгая схема common random numbers.

In [ ]:
def rollout(env,agent,epsilon,learn=False,trajectory=False):
    s = env.reset()
    path=[env.pos.copy()]
    total=0.; collisions=0; initial=env.battery; energy_spent=0.
    while True:
        a=agent.act(s,epsilon)
        ns,r,done,info=env.step(a)
        if learn: agent.update(s,a,r,ns,done)
        total+=r
        collisions+=int(info['collision'])
        # расход энергии — суммарные положительные уменьшения батареи,
        # без вычитания энергии, полученной на станции
        v=env.actions[a][1]
        energy_spent+=0.018*v*v+0.003*np.dot(env.wind,env.wind)+0.002*abs(v)
        if trajectory: path.append(env.pos.copy())
        s=ns
        if done: break
    return dict(reward=total,steps=env.t,success=info['success'],
                collisions=collisions,collision_episode=collisions>0,
                battery=env.battery,energy=energy_spent*100,
                path=np.array(path) if trajectory else None)

def train(agent,seed,episodes=TRAIN_EPISODES):
    env=DroneEnv(seed)
    rows=[]
    for ep in range(episodes):
        epsilon=max(0.01,1.0*(EPS_DECAY**ep))
        row=rollout(env,agent,epsilon,learn=True)
        rows.append({k:v for k,v in row.items() if k!='path'})
        if (ep+1)%3000==0: print(f'  {ep+1}/{episodes}: recent success={np.mean([x["success"] for x in rows[-300:]]):.1%}')
    return pd.DataFrame(rows)

def evaluate(agent,seed,episodes=TEST_EPISODES,paths=4):
    env=DroneEnv(seed)
    rows=[rollout(env,agent,0.005,trajectory=(i<paths)) for i in range(episodes)]
    return pd.DataFrame(rows)

agents={'Rational':TabularAgent(seed=SEED+1),
        'Prospect Theory':ProspectAgent(seed=SEED+2),
        'Risk-sensitive':RiskAgent(seed=SEED+3,eta=ETA)}
train_logs={}
for name,agent in agents.items():
    print('Training',name)
    train_logs[name]=train(agent,seed=SEED+100)
test_logs={name:evaluate(agent,seed=SEED+200) for name,agent in agents.items()}

In [ ]:
def metrics(df):
    wins=df[df.success]
    return dict(success_pct=100*df.success.mean(),
        mean_time_success=wins.steps.mean() if len(wins) else np.nan,
        mean_energy_pct=df.energy.mean(),
        collision_episodes_pct=100*df.collision_episode.mean(),
        mean_collisions=df.collisions.mean(),
        mean_reward=df.reward.mean())
summary=pd.DataFrame({name:metrics(df) for name,df in test_logs.items()}).T
display(summary.round(3))

## 3. Визуализации

По требованиям: сглаженная кривая обучения (150), CDF времени **успешных** эпизодов, тепловая карта конечного заряда (частоты по бинам заряда и алгоритмам) и 4 траектории каждого агента. Для отсутствующих успехов CDF пропускается с пояснением.

In [ ]:
plt.figure(figsize=(11,5))
for name,df in train_logs.items():
    plt.plot(df.reward.rolling(150,min_periods=150).mean(),label=name)
plt.xlabel('Эпизод');plt.ylabel('Награда, среднее по 150')
plt.title('Кривые обучения');plt.grid(alpha=.3);plt.legend();plt.show()

plt.figure(figsize=(9,5))
for name,df in test_logs.items():
    times=np.sort(df.loc[df.success,'steps'].to_numpy())
    if len(times): plt.step(times,np.arange(1,len(times)+1)/len(times),where='post',label=name)
    else: print('Нет успешных эпизодов для CDF:',name)
plt.xlabel('Шаги до успеха');plt.ylabel('CDF | успех')
plt.title('Распределение времени успеха');plt.grid(alpha=.3);plt.legend();plt.show()

bins=np.linspace(0,1,21)
heat=np.array([np.histogram(df.battery,bins=bins)[0] for df in test_logs.values()])
fig,ax=plt.subplots(figsize=(11,3.5))
im=ax.imshow(heat,aspect='auto',cmap='viridis')
ax.set_yticks(range(len(test_logs)),list(test_logs))
ax.set_xticks(range(0,20,2),[f'{bins[i]:.1f}' for i in range(0,20,2)])
ax.set_xlabel('Финальный заряд (нижняя граница бина)')
ax.set_title('Heatmap: количество тестовых эпизодов')
fig.colorbar(im,ax=ax,label='Эпизодов');plt.tight_layout();plt.show()

fig,axes=plt.subplots(1,3,figsize=(17,4.5),sharex=True,sharey=True)
for ax,(name,df) in zip(axes,test_logs.items()):
    for xmin,xmax,ymin,ymax in CFG.obstacles:
        ax.add_patch(plt.Rectangle((xmin,ymin),xmax-xmin,ymax-ymin,alpha=.4,color='gray'))
    ax.add_patch(plt.Circle(CFG.station,CFG.radius,color='green',alpha=.25))
    for i,path in enumerate(df.path.iloc[:4]):
        ax.plot(path[:,0],path[:,1],lw=1,label=f'Траектория {i+1}')
        ax.plot(path[0,0],path[0,1],'.',color='black',ms=4)
    ax.set(xlim=(0,24),ylim=(0,12),title=name,xlabel='x')
    ax.grid(alpha=.2);ax.legend(fontsize=7)
axes[0].set_ylabel('y');plt.tight_layout();plt.show()

## 4. Статистическое сравнение и исследовательский вывод

Попарный двухсторонний Welch t-test по расходу энергии (числовая метрика, доступная для **всех** эпизодов, включая неудачные). Порог p<0.05. Нулевые гипотезы: равенство математических ожиданий. Поскольку выполняются три попарных теста, дополнительно выводится решение с поправкой Бонферрони (0.05/3). Значимость сама по себе не доказывает практическую полезность; интерпретируйте вместе с успехом и коллизиями.

In [ ]:
tests=[]
for (a,da),(b,db) in combinations(test_logs.items(),2):
    t,p=ttest_ind(da.energy.to_numpy(),db.energy.to_numpy(),
                  equal_var=False,nan_policy='omit')
    tests.append(dict(agent_A=a,agent_B=b,t_stat=t,p_value=p,
                      significant_005=bool(p<.05),
                      significant_bonferroni=bool(p<.05/3),
                      energy_diff_pct_points=da.energy.mean()-db.energy.mean()))
stats=pd.DataFrame(tests)
display(stats)
print('Лучший по успеху:',summary.success_pct.idxmax())
print('Минимум среднего расхода:',summary.mean_energy_pct.idxmin())
print('Минимум доли эпизодов с коллизиями:',summary.collision_episodes_pct.idxmin())
print('Важно: при нулевом числе успехов среднее время до успеха неопределено.')

## 5. Проверки корректности

Эти тесты быстрые и не требуют повторного обучения. Проверяют геометрию, размеры дискретизации, границы батареи, постоянство ветра в периоде и отсутствие прохода через препятствие. Основные результаты выше — фактические после запуска, а не заранее заданные.

In [ ]:
assert len(DroneEnv().actions)==16
assert 14*14*12*4==9408
assert segment_hits_rect(np.array([9.,3.]),np.array([12.,3.]),CFG.obstacles[0])
assert not segment_hits_rect(np.array([1.,1.]),np.array([2.,1.]),CFG.obstacles[0])
e=DroneEnv(77)
e.reset()
wind=e.wind.copy()
for i in range(14):
    _,_,done,info=e.step(12)  # удержание позиции; ветер всё ещё действует
    assert np.allclose(e.wind,wind)
    assert 0<=e.battery<=1
    assert 0<=e.pos[0]<=24 and 0<=e.pos[1]<=12
    assert all(0<=v<lim for v,lim in zip(e.observe(),(14,14,12,4)))
assert not done
print('Быстрые проверки пройдены.')
print('Проверка обучения:',{name:len(df) for name,df in train_logs.items()})
print('Проверка тестирования:',{name:len(df) for name,df in test_logs.items()})
assert all(len(df)==TRAIN_EPISODES for df in train_logs.values())
assert all(len(df)==TEST_EPISODES for df in test_logs.values())

## 6. Интерпретация и ограничения

После выполнения сопоставьте таблицу метрик, CDF, кривые и траектории. Возможен компромисс между вероятностью успеха, скоростью и затратами энергии. Динамика движения и набор служебных действий в методичке недоопределены; принятые допущения перечислены в начале. Тестирование на одном seed — учебный эксперимент, а не окончательная оценка обобщающей способности. Для более надёжных выводов следует повторить обучение на нескольких независимых seed и сообщить интервалы неопределённости. Экспоненциальная полезность чувствительна к редким крупным штрафам, поэтому выбран η=0.5 — минимальное разрешённое положительное значение. Сравнение с η=1 и η=2 может быть отдельным дополнительным экспериментом.

**Важное ограничение:** в текущей версии расход энергии в отчёте рассчитывается по формуле потребления до зарядки, а не как разница начального и конечного заряда. Это позволяет сравнивать затраты при разных периодах зарядки.